# Imports & Paths

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import ast
from pathlib import Path
import pickle
from sklearn.multioutput import MultiOutputClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import RidgeClassifier
from catboost import CatBoostClassifier
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

BASE_DIR = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
DATA_DIR = BASE_DIR / "data"

# Load the input data

## Train / test splits (OHE labels + Mordred features)

In [ ]:
OHE_train = pd.read_csv(DATA_DIR / "OHE_train_dataset_All_Food.csv", sep=';', header=0, index_col=0)
OHE_test = pd.read_csv(DATA_DIR / "OHE_test_dataset_All_Food.csv",  sep=';', header=0, index_col=0)

y_train = np.array([np.array(ast.literal_eval(f), dtype=int) for f in OHE_train['OneHotEncoding']])
y_test = np.array([np.array(ast.literal_eval(f), dtype=int) for f in OHE_test['OneHotEncoding']])

X_train_index = pd.read_csv(DATA_DIR / 'X_train_features_Mordred.csv', sep=',')
X_test_index = pd.read_csv(DATA_DIR / 'X_test_features_Mordred.csv',  sep=',')
X_test = np.delete(X_test_index,  0, 1)
X_train = np.delete(X_train_index, 0, 1)


## Full dataset for 5-fold CV

In [ ]:
y_full = np.vstack([y_train, y_test])
X_full = np.vstack([X_train, X_test])

print(f"Full dataset: {X_full.shape[0]} samples, {X_full.shape[1]} features, "
      f"{y_full.shape[1]} labels")

# Utility helpers

In [ ]:
def clone_estimator(est):
    from sklearn.base import clone
    try:
        return clone(est)
    except Exception:
        import copy
        return copy.deepcopy(est)

def cv_roc_pr(estimator, X, y, n_splits=5, random_state=42):
    skf = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    roc_data, pr_data = [], []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"  Fold {fold}/{n_splits} …")
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        clf = clone_estimator(estimator)
        clf.fit(X_tr, y_tr)

        if hasattr(clf, 'predict_proba'):
            proba = clf.predict_proba(X_val)
        else:
            dec = np.column_stack([
                est.decision_function(X_val) for est in clf.estimators_
            ])
            proba = (dec - dec.min(axis=0)) / (dec.max(axis=0) - dec.min(axis=0) + 1e-9)

        if isinstance(proba, list):
            scores_flat = np.column_stack([p[:, 1] for p in proba]).flatten()
        else:
            scores_flat = proba.flatten()

        y_flat = y_val.flatten()

        fpr,  tpr,  _ = roc_curve(y_flat, scores_flat)
        prec, rec,  _ = precision_recall_curve(y_flat, scores_flat)

        roc_data.append((fpr, tpr, auc(fpr, tpr)))
        pr_data.append((rec, prec, auc(rec, prec)))

    return roc_data, pr_data

# Default models – 5-fold CV on full dataset

In [ ]:
default_models = {
    "ExtraTreesClassifier": MultiOutputClassifier(ExtraTreesClassifier(random_state=42)),
    "KNeighborsClassifier": MultiOutputClassifier(KNeighborsClassifier()),
    "RidgeClassifier": MultiOutputClassifier(RidgeClassifier()),
    "RandomForestClassifier": MultiOutputClassifier(RandomForestClassifier(random_state=42)),
    "CatBoost": OneVsRestClassifier(CatBoostClassifier(random_state=42, verbose=False,thread_count=-1)),
}

default_cv_results = {}
for name, model in default_models.items():
    print(f"Running 5-fold CV for {name} …")
    roc_data, pr_data = cv_roc_pr(model, X_full, y_full, n_splits=5)
    default_cv_results[name] = (roc_data, pr_data)
    print(f"  Mean ROC-AUC = {np.mean([r[2] for r in roc_data]):.4f}  |  "
          f"Mean PR-AUC = {np.mean([p[2] for p in pr_data]):.4f}")

#with open(BASE_DIR / "cv_results_default.pkl", "wb") as f:
#    pickle.dump(default_cv_results, f)

# Optimized models – 5-fold CV on full dataset

In [ ]:
optimized_models = {
    "RandomForest (opt)": MultiOutputClassifier(RandomForestClassifier(
        random_state=42,
        n_estimators=1100,
        min_samples_leaf=3,
        min_samples_split=2,
    )),
    "CatBoost (opt)": OneVsRestClassifier(CatBoostClassifier(
        n_estimators=1000,
        depth=6,
        learning_rate=0.01,
        loss_function='Logloss',
        random_seed=42,
        thread_count=-1,
        verbose=False,
    )),
}

optimized_cv_results = {}
for name, model in optimized_models.items():
    print(f"Running 5-fold CV for {name} …")
    roc_data, pr_data = cv_roc_pr(model, X_full, y_full, n_splits=5)
    optimized_cv_results[name] = (roc_data, pr_data)
    print(f"  Mean ROC-AUC = {np.mean([r[2] for r in roc_data]):.4f}  |  "
          f"Mean PR-AUC = {np.mean([p[2] for p in pr_data]):.4f}")

#with open(BASE_DIR / "cv_results_optimized.pkl", "wb") as f:
#    pickle.dump(optimized_cv_results, f)

# Train final optimized models on train split & predict on test split

In [ ]:
RFCLassifier = MultiOutputClassifier(RandomForestClassifier(
    random_state=42,
    n_estimators=1100,
    min_samples_leaf=3,
    min_samples_split=2,
))
RFCLassifier.fit(X_train, y_train)
y_scores_RF = RFCLassifier.predict_proba(X_test)
y_pred_RF = RFCLassifier.predict(X_test)

Cat = OneVsRestClassifier(CatBoostClassifier(
    n_estimators=1000,
    depth=6,
    learning_rate=0.01,
    loss_function='Logloss',
    random_seed=42,
    verbose=False,
))
Cat.fit(X_train, y_train)
y_scores_Cat = Cat.predict_proba(X_test)
y_pred_Cat   = Cat.predict(X_test)

# Plotting

In [ ]:
PALETTE = {
    "CatBoost": "#b8974b",
    "CatBoost (opt)": "#b8974b",
    "ExtraTreesClassifier": "#999999",
    "KNeighborsClassifier": "#777777",
    "RidgeClassifier": "#555555",
    "RandomForestClassifier": "#c977aa",
    "RandomForest (opt)": "#c977aa",
}
FS = 14

def _mean_curve_roc(roc_data):
    base_fpr = np.linspace(0, 1, 300)
    tprs = [np.interp(base_fpr, fpr, tpr) for fpr, tpr, _ in roc_data]
    return base_fpr, np.mean(tprs, axis=0), np.std(tprs, axis=0)

def _mean_curve_pr(pr_data):
    base_rec = np.linspace(0, 1, 300)
    precs = [np.interp(base_rec, np.flip(rec), np.flip(prec)) for rec, prec, _ in pr_data]
    return base_rec, np.mean(precs, axis=0), np.std(precs, axis=0)

def plot_cv_roc_pr(cv_results, save_path=None):

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    for ax in axes:
        ax.set_box_aspect(1)
        ax.set_facecolor("white")
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_linewidth(0.8)

    ax_roc, ax_pr = axes

    for idx, (name, (roc_data, pr_data)) in enumerate(cv_results.items()):
        color = PALETTE.get(name, "#888888")
        aucs_roc = [r[2] for r in roc_data]
        aucs_pr  = [p[2] for p in pr_data]

        # ROC
        base_fpr, mean_tpr, std_tpr = _mean_curve_roc(roc_data)
        ax_roc.plot(base_fpr, mean_tpr, color=color, lw=2,
                    label=f"{name}  (AUC {np.mean(aucs_roc):.3f} ± {np.std(aucs_roc):.3f})")
        ax_roc.fill_between(base_fpr, mean_tpr - std_tpr, mean_tpr + std_tpr,
                            color=color, alpha=0.15)

        # PR
        base_rec, mean_prec, std_prec = _mean_curve_pr(pr_data)
        ax_pr.plot(base_rec, mean_prec, color=color, lw=2,label=f"{name}  (AUC {np.mean(aucs_pr):.3f} ± {np.std(aucs_pr):.3f})")
        ax_pr.fill_between(base_rec, mean_prec - std_prec, mean_prec + std_prec,color=color, alpha=0.15)

    ax_roc.plot([0, 1], [0, 1], '--', color='#AAAAAA', lw=1)

    for ax, xlabel, ylabel in [
        (ax_roc, "False Positive Rate", "True Positive Rate"),
        (ax_pr, "Recall", "Precision"),
    ]:
        ax.set_xlabel(xlabel, fontsize=FS)
        ax.set_ylabel(ylabel, fontsize=FS)
        ax.tick_params(labelsize=FS)
        ax.legend(loc="best", fontsize=FS - 1, frameon=False)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=600, bbox_inches='tight')
    plt.show()

In [ ]:
#with open(BASE_DIR / "cv_results_default.pkl", "rb") as f:
#    default_cv_results = pickle.load(f)

#with open(BASE_DIR / "cv_results_optimized.pkl", "rb") as f:
#    optimized_cv_results = pickle.load(f)

plot_cv_roc_pr(default_cv_results)
plot_cv_roc_pr(optimized_cv_results)

# Classification metrics summary (train/test split, optimized models)

In [ ]:
lists = pd.read_csv(DATA_DIR / 'Total_dataset_All_Food.csv', sep=';', header=0)["Labels"].apply(ast.literal_eval)
unique_items = list(set(item for sublist in lists for item in sublist))

OHE_obj = pd.DataFrame(
    OHE_test["OneHotEncoding"].apply(ast.literal_eval).tolist(),
    index=OHE_test["SMILES"],
    columns=unique_items,
)
OHE_num = OHE_obj.to_numpy().flatten()

model_preds = {
    "RandomForest (opt)": np.column_stack([p[:, 1] for p in RFCLassifier.predict_proba(X_test)]).flatten(),
    "CatBoost (opt)": Cat.predict_proba(X_test).flatten(),
}
model_bin_preds = {
    "RandomForest (opt)": y_pred_RF,
    "CatBoost (opt)": y_pred_Cat,
}

average_type = "macro"
print(f"{'Model':<22} {'F1':<14} {'Avg Score (AP)':<18} {'Precision':<20} {'Recall'}")
print("-" * 90)

metrics_rows = []
for label, y_pred in model_bin_preds.items():
    y_score_flat = model_preds[label]

    f1 = f1_score(y_test.flatten(),  y_pred.flatten(), average=average_type, zero_division=0)
    avg_score = average_precision_score(OHE_num, y_score_flat, average=average_type)
    precision = precision_score(y_test.flatten(), y_pred.flatten(), average=average_type, zero_division=0)
    recall = recall_score(y_test.flatten(),    y_pred.flatten(), average=average_type, zero_division=0)

    metrics_rows.append({
        "Model": label,
        f"F1 ({average_type})": round(f1, 4),
        "Avg Score (AP)": round(avg_score, 4),
        f"Precision ({average_type})": round(precision, 4),
        f"Recall ({average_type})": round(recall, 4),
    })
    print(f"{label:<22} {f1:<14.4f} {avg_score:<18.4f} {precision:<20.4f} {recall:.4f}")

metrics_df = pd.DataFrame(metrics_rows).set_index("Model")
display(metrics_df.style
    .format("{:.4f}")
    .set_caption(f"Classification Metrics — {average_type}-averaged")
    .set_table_styles([{"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold")]}])
    .background_gradient(cmap="YlGn", axis=0)
)